### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

page_content: a string representing the content;
metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [2]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="React is a frontend library",
        metadata={"source": "notes", "topic": "frontend"}
    ),
    Document(
        page_content="FastAPI is used for backend APIs",
        metadata={"source": "notes", "topic": "backend"}
    )
]
docs

[Document(metadata={'source': 'notes', 'topic': 'frontend'}, page_content='React is a frontend library'),
 Document(metadata={'source': 'notes', 'topic': 'backend'}, page_content='FastAPI is used for backend APIs')]

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

model=ChatGroq(groq_api_key=groq_api_key, model="Llama-3.1-Pro-Mistral-8K-GROQ")


In [6]:
#vectorStores
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(docs, embeddings)
vectorstore.similarity_search("What is used for backend APIs?")
await vectorstore.asimilarity_search("What is used for backend APIs?")



[Document(id='5b56ef03-f6e3-429b-9340-a938d4792942', metadata={'source': 'notes', 'topic': 'backend'}, page_content='FastAPI is used for backend APIs'),
 Document(id='a9664283-7562-4191-9d14-8b0ebfbd1bc9', metadata={'topic': 'backend', 'source': 'notes'}, page_content='FastAPI is used for backend APIs'),
 Document(id='3a0190ab-3bfe-4f6c-9d30-2cbc79e4ecc7', metadata={'source': 'notes', 'topic': 'frontend'}, page_content='React is a frontend library'),
 Document(id='d9552c35-9a77-439e-9e93-6e61bb12e72a', metadata={'source': 'notes', 'topic': 'frontend'}, page_content='React is a frontend library')]

Retrievers

LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [7]:
retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":1}  )
retriever.batch(["What is used for backend APIs?", "What is used for frontend development?"])

[[Document(id='a9664283-7562-4191-9d14-8b0ebfbd1bc9', metadata={'topic': 'backend', 'source': 'notes'}, page_content='FastAPI is used for backend APIs')],
 [Document(id='d9552c35-9a77-439e-9e93-6e61bb12e72a', metadata={'topic': 'frontend', 'source': 'notes'}, page_content='React is a frontend library')]]